In [1]:
import jax
import jax.random as jrandom
import jax.tree_util as jtu
import jax.numpy as jnp
from jax import lax

from diffrax import diffeqsolve, ControlTerm, Euler, MultiTerm, ODETerm, SaveAt, VirtualBrownianTree, Tsit5, \
    VirtualBrownianTree, STLAControlTerm, STLAMultiTerm, SEA, ShARK, PIDController, LevyVal, ALIGN

import diffrax
import math
import matplotlib.pyplot as plt

def drift(t, y, args):
    gamma, u, grad_f = args
    x, v = y[:2], y[2:]
    d_x = v
    d_v = -gamma * v - u * grad_f(x)
    d_y = jnp.array([d_x, d_v], dtype=jnp.float32).flatten()
    return d_y


def diffusion(t, y, args):
    gamma, u, _ = args
    d_v = jnp.sqrt(2 * gamma * u) * jnp.ones((2,), dtype=y.dtype)
    d_y = jnp.concatenate((jnp.zeros((2, 2), dtype=jnp.float32), jnp.diag(d_v)), axis=0)
    return d_y

def L2_distance(ys1: jax.Array, ys2: jax.Array):
    assert ys1.shape == ys2.shape
    n = ys1.shape[0]
    square_dist = jnp.square(ys1 - ys2)
    avg = 1/n * jnp.sum(square_dist)
    return jnp.sqrt(avg)

args = (0.3, 5.0, lambda x: x)
y0 = jnp.array([0, 0, 0, 0], dtype=jnp.float32)
sde = (drift, diffusion, args, y0)
t0, t1 = 0.3, 15
num_samples = 1000

In [2]:
print(jax.devices())

[cuda(id=0)]


In [3]:
def solutions(t0, t1, keys, sde, dt0, solver):
    drift, diffusion, args, y0 = sde
    saveat = SaveAt(ts=[t1])
    ode_term = ODETerm(drift)
    def end_value(key):
        path = diffrax.VirtualBrownianTree(t0=t0, t1=t1, shape=(2,), tol=2**-9, key=key, compute_stla=True)
        terms = MultiTerm(ode_term, ControlTerm(diffusion, path))
        sol = diffeqsolve(terms, solver, t0, t1, dt0=dt0, y0=y0, args=args, saveat=saveat)
        return sol.ys[0]
    
    return jax.vmap(end_value)(keys)
    

In [0]:
keys = jrandom.split(jrandom.PRNGKey(2), num=num_samples)
solsALIGN4 = solutions(t0, t1, keys, sde, dt0=0.4, solver=ALIGN())
solsALIGN2 = solutions(t0, t1, keys, sde, dt0=0.2, solver=ALIGN())
solsALIGN1 = solutions(t0, t1, keys, sde, dt0=0.1, solver=ALIGN())
solsALIGN_5 = solutions(t0, t1, keys, sde, dt0=0.05, solver=ALIGN())

In [7]:
solsALIGN16 = solutions(t0, t1, keys, sde, dt0=1.6, solver=ALIGN())
solsALIGN8 = solutions(t0, t1, keys, sde, dt0=0.8, solver=ALIGN())

recomputing coeffs for h = 1.6000001430511475
recomputing coeffs for h = 0.29999828338623047
recomputing coeffs for h = 0.800000011920929
recomputing coeffs for h = 0.29999732971191406


In [9]:
solsPrecise = solutions(t0, t1, keys, sde, dt0=0.005, solver=ShARK())

In [13]:
errs = [float(L2_distance(sols, solsPrecise)) for sols in [solsALIGN8, solsALIGN4, solsALIGN2, solsALIGN1, solsALIGN_5]]

print(errs)
print([x/y for x, y in zip(errs[:-1], errs[1:])])

[4.7140727043151855, 1.1696972846984863, 0.28166714310646057, 0.06910312920808792, 0.021439578384160995]
[4.030164698151228, 4.152764400554809, 4.076040352069814, 3.2231570961833618]
